# 04 · Hình và bảng cho báo cáo

Gom mọi thứ đã chạy thành sản phẩm nộp. Không huấn luyện gì thêm ở đây.

Người phụ trách: **SV B**.

In [ ]:
# Chạy được cả trên Colab lẫn máy cá nhân
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("unet-kvasir").exists():
    # !git clone <repo cua nhom> unet-kvasir
    pass
ROOT = Path("unet-kvasir") if Path("unet-kvasir").exists() else Path("..")
os.chdir(ROOT.resolve())
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

from src import *
print("Thư mục làm việc:", Path.cwd())
print("Thiết bị:", get_device())

In [ ]:
import pandas as pd

cfg = Config()
df = RunLogger(cfg.log_csv).to_dataframe()
num = [c for c in df.columns if c.startswith(("test_", "best_val", "n_params"))]
df[num] = df[num].astype(float)
print(f"Tổng số lượt chạy trong log: {len(df)}")
df[["run_id", "loss_name", "up_mode", "skip_mode", "seed", "test_dice"]].round(4)

## Bảng tổng hợp cho báo cáo

In [ ]:
summary = (df[["loss_name", "up_mode", "skip_mode", "seed", "n_params",
                "test_dice", "test_iou", "test_precision", "test_recall",
                "train_time_min"]]
           .sort_values("test_dice", ascending=False).round(4))
print(summary.to_markdown(index=False))

## Lưới ảnh so sánh (yêu cầu tối thiểu 10 ảnh)

Đề bài yêu cầu: ảnh gốc / mask thật / dự đoán của 2–3 cấu hình tốt nhất và
1 cấu hình tệ nhất.

In [ ]:
from src.viz import comparison_grid

best_rows = df.nlargest(3, "test_dice")
worst_row = df.nsmallest(1, "test_dice").iloc[0]

def cfg_from_row(row):
    return Config(loss_name=row["loss_name"], up_mode=row["up_mode"],
                  skip_mode=row["skip_mode"], seed=int(row["seed"]))

models = {}
for _, row in best_rows.iterrows():
    c = cfg_from_row(row)
    models[f"{row.loss_name}/{row.up_mode}/{row.skip_mode}"] = \
        load_best(build_model(c), c.ckpt_path, get_device())
c = cfg_from_row(worst_row)
models[f"TỆ NHẤT: {worst_row.skip_mode}"] = \
    load_best(build_model(c), c.ckpt_path, get_device())

splits = load_splits(cfg.split_dir)
test_ds = KvasirSegDataset(cfg.data_root, splits["test"],
                           SegTransform(cfg.image_size, train=False))

In [ ]:
fig = comparison_grid(test_ds, models, indices=range(10), device=get_device(),
                      save_path=f"{cfg.fig_dir}/comparison_grid.png")

## Ảnh mô hình làm tệ nhất

Nguyên liệu cho mục *Phân tích và thảo luận*. Nhìn kỹ xem chúng có điểm chung
gì: polyp quá nhỏ, biên mờ, có bọt khí, ánh sáng chói, hay nhiều polyp.

In [ ]:
from src.viz import worst_case_indices
from torch.utils.data import DataLoader

best_name = list(models)[0]
loader = DataLoader(test_ds, batch_size=8, shuffle=False)
worst = worst_case_indices(models[best_name], loader, get_device(), k=6)
print("Chỉ số 6 ảnh tệ nhất:", worst)

fig = comparison_grid(test_ds, {best_name: models[best_name]}, indices=worst,
                      device=get_device(),
                      save_path=f"{cfg.fig_dir}/worst_cases.png")

## Danh sách kiểm trước khi nộp

- [ ] `logs/runs.csv` có đủ mọi lượt chạy, kể cả lượt hỏng
- [ ] Mọi số trong báo cáo đều đọc từ CSV, không gõ tay
- [ ] `README.md` chạy lại được từ đầu trên máy sạch
- [ ] `requirements.txt` khớp với môi trường thật
- [ ] Bảng phân công có tỉ lệ đóng góp cộng lại đúng 100%
- [ ] Mục *Khai báo sử dụng công cụ AI* ở cuối báo cáo
- [ ] Hình nào cũng có chú thích và được nhắc tới trong nội dung